# 06 - Hedonic Regression Data Prep (Descriptive)

Builds the analysis-ready dataset for the descriptive hedonic regression in R.

## Inputs
- `../data/processed/nri_panel_smooth.pkl` - NRI county-month panel (interpolated)
- `../data/processed/NRI_Long.csv` - source NRI by vintage (provides population, area, resl_value, sovi_value)
- `../data/raw/zillow_county_smooth_zhvi.csv` - smoothed/SA Zillow ZHVI (county-month)
- `../data/NRI_2023.csv` and `../data/NRI_2025.csv` - source NRI tables (used to derive coastal flag from CFLD_EALT)

## Output
- `../data/processed/hedonic_dataset.csv`
- `../data/processed/hedonic_dataset.pkl`

## Sample
- Window: **Nov 2020 - Dec 2025** (full available)
- Frequency: monthly county-month observations
- `cdc_svi_regime` dummy = 1 from March 2023 onward (NRI v1.19 SoVI methodology change)

## NRI components carried
- `eal_valt` (continuous): Expected Annual Loss
- `risk_value` (continuous): Risk Index Value (= eal_valt * crf_value)
- `crf_value` (continuous): Community Risk Factor (= f(SoVI / Resilience))
- `resl_value` (continuous): raw BRIC resilience index
- `resl_score` (percentile): national-rank resilience
- `sovi_score` (percentile): national-rank social vulnerability
  - HVRI SoVI percentile pre-March 2023; CDC/ATSDR SVI percentile thereafter
- `sovi_value` (continuous, pre-2023 only): HVRI SoVI raw index. Null after 2023 by design.
- `population`, `area` (vintage-matched, used for population density)

## Coastal flag
Defined as `CFLD_EALT > 0` from any NRI vintage. NRI computes coastal-flooding EAL only for counties
with coastline exposure (Atlantic, Pacific, Gulf, Great Lakes), so this matches NOAA's coastal-shoreline
county definition without parsing the NOAA PDF list.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load NRI long format with extras
Pull the columns we need from `NRI_Long.csv`. We use this rather than `nri_panel.pkl` because
the panel pipeline drops `population`, `area`, `resl_value`, and `sovi_value`.

In [3]:
NRI_LONG_COLS = [
    'nri_id', 'stateabbrv', 'county',
    'population', 'area', 'buildvalue', 'agrivalue',
    'risk_value', 'eal_valt', 'crf_value',
    'sovi_score', 'resl_score', 'resl_value',
    'year'  # NRI vintage
]

nri_long = pd.read_csv('../data/processed/NRI_Long.csv', usecols=NRI_LONG_COLS)
nri_long = nri_long.rename(columns={'year': 'nri_vintage'})
nri_long['stcofips'] = nri_long['nri_id'].str[1:]

# SOVI_VALUE exists only in 2020 and 2021 source files (HVRI methodology).
# Pull it directly from those source NRI tables and merge in.
sovi_val_frames = []
for vintage in [2020, 2021]:
    src_path = Path(f'../data/NRI_{vintage}.csv')
    if src_path.exists():
        s = pd.read_csv(src_path, usecols=['STCOFIPS', 'SOVI_VALUE'], low_memory=False)
        s['stcofips']    = s['STCOFIPS'].astype(str).str.zfill(5)
        s['nri_vintage'] = vintage
        s['sovi_value']  = s['SOVI_VALUE']
        sovi_val_frames.append(s[['stcofips', 'nri_vintage', 'sovi_value']])

if sovi_val_frames:
    sovi_val = pd.concat(sovi_val_frames, ignore_index=True)
    nri_long = nri_long.merge(sovi_val, on=['stcofips', 'nri_vintage'], how='left')
    print(f'sovi_value attached for {sovi_val["stcofips"].nunique():,} counties x {sovi_val["nri_vintage"].nunique()} vintages')
else:
    nri_long['sovi_value'] = np.nan
    print('No source NRI files found for SOVI_VALUE; column will be all NaN.')

print(f'\nNRI_Long rows: {len(nri_long):,}')
print(f'Vintages: {sorted(nri_long.nri_vintage.unique())}')
print(f'Counties per vintage:')
print(nri_long.groupby("nri_vintage").size())
print(f'\nsovi_value null rate by vintage (expect 100% in 2023, 2025):')
print(nri_long.groupby("nri_vintage")["sovi_value"].apply(lambda s: f"{s.isna().mean():.1%}"))

sovi_value attached for 3,142 counties x 2 vintages

NRI_Long rows: 12,576
Vintages: [np.int64(2020), np.int64(2021), np.int64(2023), np.int64(2025)]
Counties per vintage:
nri_vintage
2020    3144
2021    3144
2023    3144
2025    3144
dtype: int64

sovi_value null rate by vintage (expect 100% in 2023, 2025):
nri_vintage
2020      0.3%
2021      0.3%
2023    100.0%
2025    100.0%
Name: sovi_value, dtype: object


## 2. Build county-month NRI panel using interpolated scores

Use the existing `nri_panel_smooth.pkl` for the interpolated month-level scores (`risk_value`,
`eal_valt`, `crf_value`, `resl_score`, `sovi_score`), and merge in vintage-level extras
(`population`, `area`, `resl_value`, `sovi_value`) by joining on `stcofips` + `nri_vintage`.

Vintage-level extras don't get monthly interpolation - they're step functions that change at
each vintage anchor. That's the right behavior for variables like population that are reported
as snapshots.

In [4]:
panel = pd.read_pickle('../data/processed/nri_panel_smooth.pkl')
# Build a proper month-start datetime from storm_year + month
panel = panel.rename(columns={'month': 'month_num'})
panel['month'] = pd.to_datetime(
    panel['storm_year'].astype(str) + '-' + panel['month_num'].astype(str).str.zfill(2) + '-01'
)
print(f'Panel shape (raw): {panel.shape}')
print(f'Columns: {panel.columns.tolist()}')
panel.head(3)

Panel shape (raw): (226368, 15)
Columns: ['stcofips', 'state_fips', 'county_fips', 'stateabbrv', 'county', 'storm_year', 'month_num', 'nri_vintage', 'resl_score', 'resl_value', 'risk_value', 'eal_valt', 'sovi_score', 'crf_value', 'month']


,stcofips,state_fips,county_fips,stateabbrv,county,storm_year,month_num,nri_vintage,resl_score,resl_value,risk_value,eal_valt,sovi_score,crf_value,month
0,01001,01,001,AL,Autauga,2020,1,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,2020-01-01
1,01001,01,001,AL,Autauga,2020,2,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,2020-02-01
2,01001,01,001,AL,Autauga,2020,3,2020,55.5298,2.77649,2.602012e+06,3.311627e+06,25.857312,0.78572,2020-03-01


In [5]:
# Vintage-level extras to merge in
# resl_value is already in nri_panel_smooth; only merge in extras that aren't there
VINTAGE_EXTRAS = ['stcofips', 'nri_vintage', 'population', 'area', 'sovi_value']
extras = nri_long[VINTAGE_EXTRAS].drop_duplicates(['stcofips', 'nri_vintage'])

panel = panel.merge(extras, on=['stcofips', 'nri_vintage'], how='left')

print(f'Panel shape after extras merge: {panel.shape}')
print(f'Missing population: {panel.population.isna().sum():,}')
print(f'Missing area: {panel.area.isna().sum():,}')
print(f'Missing resl_value: {panel.resl_value.isna().sum():,}')
print(f'Missing sovi_value (expected to be all 2023+): {panel.sovi_value.isna().sum():,}')
print()
print('sovi_value null count by vintage (should be 100% null for 2023, 2025):')
print(panel.groupby('nri_vintage')['sovi_value'].apply(lambda s: f'{s.isna().mean():.1%}'))

Panel shape after extras merge: (226368, 18)
Missing population: 0
Missing area: 0
Missing resl_value: 0
Missing sovi_value (expected to be all 2023+): 107,314

sovi_value null count by vintage (should be 100% null for 2023, 2025):
nri_vintage
2020      0.3%
2021      0.3%
2023    100.0%
2025    100.0%
Name: sovi_value, dtype: object


## 3. Load and reshape Zillow ZHVI (smoothed/SA, monthly)

In [6]:
MIN_DATE = '2020-11-01'
MAX_DATE = '2025-12-31'

zillow = pd.read_csv('../data/raw/zillow_county_smooth_zhvi.csv')

META_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
             'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS']
date_cols = [c for c in zillow.columns if c not in META_COLS]

zhvi = zillow.melt(
    id_vars=['StateCodeFIPS', 'MunicipalCodeFIPS'],
    value_vars=date_cols,
    var_name='date',
    value_name='zhvi'
)
zhvi['date']        = pd.to_datetime(zhvi['date'])
zhvi['state_fips']  = zhvi['StateCodeFIPS'].astype(str).str.zfill(2)
zhvi['county_fips'] = zhvi['MunicipalCodeFIPS'].astype(str).str.zfill(3)
zhvi['stcofips']    = zhvi['state_fips'] + zhvi['county_fips']
zhvi['month']       = zhvi['date'].dt.to_period('M').dt.to_timestamp()

zhvi = zhvi[['stcofips', 'month', 'zhvi']].dropna(subset=['zhvi'])
zhvi = zhvi[(zhvi['month'] >= MIN_DATE) & (zhvi['month'] <= MAX_DATE)]

print(f'ZHVI county-month rows: {len(zhvi):,}')
print(f'Unique counties:        {zhvi.stcofips.nunique():,}')
print(f'Date range: {zhvi.month.min().date()} to {zhvi.month.max().date()}')
zhvi.head()

ZHVI county-month rows: 189,483
Unique counties:        3,073
Date range: 2020-11-01 to 2025-12-01


,stcofips,month,zhvi
768250,06037,2020-11-01,695422.882756
768251,17031,2020-11-01,248401.533255
768252,48201,2020-11-01,220042.203516
768253,04013,2020-11-01,343288.960646
768254,06073,2020-11-01,660365.387411


## 4. Derive coastal flag from NRI's coastal-flooding EAL

Counties with `CFLD_EALT > 0` in any vintage are coastal-shoreline counties by NRI's own
computation - this matches NOAA's coastal-shoreline definition (Atlantic, Pacific, Gulf,
Great Lakes) without needing to parse NOAA's PDF list. Approximately 450-480 counties
depending on vintage.

In [11]:
def coastal_from_nri(path):
    df = pd.read_csv(path, usecols=['STCOFIPS', 'CFLD_EALT'], low_memory=False)
    df['stcofips'] = df['STCOFIPS'].astype(str).str.zfill(5)
    coastal = df.loc[df['CFLD_EALT'] > 0, 'stcofips'].unique()
    return set(coastal)

# Union across vintages: a county that was ever coastal-flooded in any vintage is coastal
coastal_set = set()
for f in ['../data/NRI_2020.csv','../data/NRI_2021.csv','../data/NRI_2023.csv', '../data/NRI_2025.csv']:
    if Path(f).exists():
        coastal_set |= coastal_from_nri(f)
        print(f'{f}: cumulative coastal counties = {len(coastal_set)}')

panel['coastal'] = panel['stcofips'].isin(coastal_set).astype(int)
print(f'\nCoastal county-months: {panel.coastal.sum():,} of {len(panel):,}')
print(f'Distinct coastal counties: {panel.loc[panel.coastal == 1, "stcofips"].nunique()}')

../data/NRI_2020.csv: cumulative coastal counties = 378
../data/NRI_2021.csv: cumulative coastal counties = 378
../data/NRI_2023.csv: cumulative coastal counties = 504
../data/NRI_2025.csv: cumulative coastal counties = 533

Coastal county-months: 33,984 of 226,368
Distinct coastal counties: 472


## 5. Join ZHVI to NRI panel and finalize

In [12]:
# Inner join: keep only county-months with both ZHVI and NRI
df = panel.merge(zhvi, on=['stcofips', 'month'], how='inner')

# Filter to the analytic window
df = df[(df['month'] >= MIN_DATE) & (df['month'] <= MAX_DATE)].copy()

# CDC SVI regime dummy: NRI v1.19 released March 2023
df['cdc_svi_regime'] = (df['month'] >= '2023-03-01').astype(int)

# Population density (people per sq mile)
df['pop_density'] = df['population'] / df['area']

# Convenience time fields
df['year']      = df['month'].dt.year
# month_num already present from panel rename
df['month_id']  = df['month'].dt.strftime('%Y-%m')
df['state']     = df['stcofips'].str[:2]

print(f'Final analytic dataset: {df.shape}')
print(f'Counties: {df.stcofips.nunique():,}')
print(f'Months:   {df.month.nunique()}')
print(f'Coastal counties in sample: {df.loc[df.coastal == 1, "stcofips"].nunique()}')

Final analytic dataset: (188925, 25)
Counties: 3,064
Months:   62
Coastal counties in sample: 458


In [13]:
# Sanity checks
assert df.duplicated(['stcofips', 'month']).sum() == 0, 'Duplicate county-month rows'
assert df['zhvi'].gt(0).all(), 'Non-positive ZHVI values present'
assert df['eal_valt'].ge(0).all(), 'Negative EAL values'
assert df['cdc_svi_regime'].isin([0, 1]).all()

# Verify Risk = EAL * CRF holds within rounding (multiplicative form)
implied_risk = df['eal_valt'] * df['crf_value']
rel_err = ((df['risk_value'] - implied_risk).abs() / df['risk_value'].replace(0, np.nan)).fillna(0)
print(f'Max relative error |risk - EAL*CRF| / risk: {rel_err.max():.2e}')
print(f'Median relative error: {rel_err.median():.2e}')
print()

# Variable summary
summary_cols = ['zhvi', 'eal_valt', 'risk_value', 'crf_value',
                'resl_value', 'resl_score', 'sovi_score', 'sovi_value',
                'population', 'area', 'pop_density', 'coastal', 'cdc_svi_regime']
df[summary_cols].describe().T.round(3)

Max relative error |risk - EAL*CRF| / risk: 3.56e-01
Median relative error: 7.67e-03



,count,mean,std,min,25%,50%,75%,max
zhvi,188925.0,2.471027e+05,1.609005e+05,42354.386,153001.774,205143.160,2.919974e+05,2.973020e+06
eal_valt,188925.0,2.638120e+07,1.217518e+08,65326.884,3308608.837,6938313.730,1.654812e+07,7.601846e+09
risk_value,188925.0,2.962962e+07,1.456154e+08,68147.919,3815741.637,8014825.950,1.886960e+07,9.180158e+09
crf_value,188925.0,1.172000e+00,2.540000e-01,0.515,0.993,1.151,1.341000e+00,2.000000e+00
resl_value,188925.0,2.608000e+00,1.590000e-01,1.950,2.496,2.605,2.715000e+00,3.234000e+00
resl_score,188925.0,5.186900e+01,2.289100e+01,0.030,37.701,53.842,6.500000e+01,1.000000e+02
sovi_score,188925.0,4.623100e+01,2.280600e+01,0.000,29.824,43.000,6.195500e+01,1.000000e+02
sovi_value,84825.0,-7.300000e-02,2.668000e+00,-9.730,-1.630,-0.020,1.460000e+00,1.410000e+01
population,188925.0,1.040616e+05,3.291144e+05,399.000,11841.000,26835.000,6.936700e+04,1.000571e+07
area,188925.0,1.044987e+03,2.260334e+03,1.985,438.199,628.565,9.442900e+02,9.578524e+04


## 6. Export

The R hedonic in `../analysis/hedonic.Rmd` reads `hedonic_dataset.csv` and merges ACS controls
(median household income, education, etc.) pulled via `tidycensus` keyed on `stcofips`.

In [14]:
OUT_COLS = [
    'stcofips', 'state', 'stateabbrv', 'county',
    'month', 'month_id', 'year', 'month_num',
    'nri_vintage', 'cdc_svi_regime',
    'zhvi',
    'eal_valt', 'risk_value', 'crf_value',
    'resl_value', 'resl_score',
    'sovi_score', 'sovi_value',
    'population', 'area', 'pop_density',
    'coastal',
]

# stateabbrv comes from nri_long extras; if it's missing in panel, pull it
if 'stateabbrv' not in df.columns:
    abbrv = nri_long[['stcofips', 'stateabbrv']].drop_duplicates('stcofips')
    df = df.merge(abbrv, on='stcofips', how='left')

out = df[OUT_COLS].sort_values(['stcofips', 'month']).reset_index(drop=True)

out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

out.to_csv(out_dir / 'hedonic_dataset.csv', index=False)
out.to_pickle(out_dir / 'hedonic_dataset.pkl')

print(f'Wrote {out_dir / "hedonic_dataset.csv"}')
print(f'Shape: {out.shape}')
print(f'Columns: {out.columns.tolist()}')

Wrote ../data/processed/hedonic_dataset.csv
Shape: (188925, 22)
Columns: ['stcofips', 'state', 'stateabbrv', 'county', 'month', 'month_id', 'year', 'month_num', 'nri_vintage', 'cdc_svi_regime', 'zhvi', 'eal_valt', 'risk_value', 'crf_value', 'resl_value', 'resl_score', 'sovi_score', 'sovi_value', 'population', 'area', 'pop_density', 'coastal']
